# 1. Анализ целей и гипотез

Курсовая работа по дисциплине «Технологии обработки больших данных».

Задача состоит в анализе глобальной рецепции экранизаций популярных литературных произведений XX–XXI веков для поиска географических и социокультурных закономерностей.

Бизнес-цель исследования — расчет маркетингового ROI. Проверяется аналитическая гипотеза: уровень зрительского принятия конкретной адаптации зависит от социокультурной специфики региона сильнее, чем от кассовых сборов фильма. Весь пайплайн строится по методологии CRISP-DM.

# 2. Обзор источников

Основным реляционным набором данных выступает датасет MovieLens (ml-32m). Здесь загружаются справочник фильмов (`movies.csv`) и файл с ключами слияния (`links.csv`). Идентификатор `tmdbId` необходим для последующего извлечения данных через REST API The Movie Database.

In [14]:
import pandas as pd

df_movies = pd.read_csv('movies.csv')
df_links = pd.read_csv('links.csv')

df_movies_linked = pd.merge(df_movies, df_links, on='movieId', how='inner')

print(f"Уникальных фильмов: {df_movies_linked.shape[0]}")
df_movies_linked.info()
display(df_movies_linked.head())

Уникальных фильмов: 87585
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  87585 non-null  int64  
 1   title    87585 non-null  object 
 2   genres   87585 non-null  object 
 3   imdbId   87585 non-null  int64  
 4   tmdbId   87461 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 3.3+ MB


,movieId,title,genres,imdbId,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844.0
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357.0
4,5,Father of the Bride Part II (1995),Comedy,113041,11862.0


Для сопоставления экранизаций с литературными первоисточниками используется датасет Goodreads. Набор данных загружается через API Kaggle и содержит оригинальные названия книг, имена авторов и оценки читателей.

In [15]:
import kagglehub

path = kagglehub.dataset_download("bahramjannesarr/goodreads-book-datasets-10m")
print(f"Путь к датасету: {path}")

Путь к датасету: C:\Users\test\.cache\kagglehub\datasets\bahramjannesarr\goodreads-book-datasets-10m\versions\18


# 3. ETL-процесс

Загрузка и предварительная очистка локального датасета Goodreads. Для снижения потребления оперативной памяти потоковая фильтрация применяется непосредственно при чтении CSV-файлов. 

Критерии фильтрации:
* Временной период: издания с 1900 года.
* Порог входа (популярность): более 1000 отзывов.
* Извлечение признаков: `Name`, `Authors`, `PublishYear`, `Rating`, `CountsOfReview`, `Description`.

В этой же ячейке выполняется загрузка и очистка справочника MovieLens (`movies.csv`, `links.csv`) для формирования целевого ключа `tmdbId` и выделения года релиза фильма.

In [16]:
import pandas as pd
import os
import glob

dataset_path = r"C:\Users\test\.cache\kagglehub\datasets\bahramjannesarr\goodreads-book-datasets-10m\versions\18"
csv_files = glob.glob(os.path.join(dataset_path, "book*.csv"))

target_cols = ['Name', 'Authors', 'PublishYear', 'Rating', 'CountsOfReview']
books_list = []

print(f"Файлов для обработки: {len(csv_files)}")

for file in csv_files:
    try:
        actual_cols = pd.read_csv(file, nrows=0).columns.tolist()
        
        usecols = [col for col in target_cols if col in actual_cols]
        
        if 'Description' in actual_cols:
            usecols.append('Description')
        elif 'description' in actual_cols:
            usecols.append('description')
            
        df_chunk = pd.read_csv(file, usecols=usecols, on_bad_lines='skip')
        
        if 'description' in df_chunk.columns:
            df_chunk = df_chunk.rename(columns={'description': 'Description'})
        if 'Description' not in df_chunk.columns:
            df_chunk['Description'] = None 
            
        df_chunk['PublishYear'] = pd.to_numeric(df_chunk['PublishYear'], errors='coerce')
        df_chunk['CountsOfReview'] = pd.to_numeric(df_chunk['CountsOfReview'], errors='coerce')
        
        mask = (df_chunk['PublishYear'] >= 1900) & (df_chunk['CountsOfReview'] > 1000)
        
        books_list.append(df_chunk[mask])
    except Exception as e:
        print(f"Ошибка чтения {os.path.basename(file)}: {e}")

df_books = pd.concat(books_list, ignore_index=True)
df_books = df_books.drop_duplicates(subset=['Name', 'Authors'])

print(f"Книг после фильтрации: {df_books.shape[0]}")
display(df_books.sort_values(by='CountsOfReview', ascending=False).head(3))

df_movies = pd.read_csv('movies.csv')
df_links = pd.read_csv('links.csv')

df_movies_linked = pd.merge(df_movies, df_links, on='movieId', how='inner')

df_movies_cleaned = df_movies_linked.dropna(subset=['tmdbId']).copy()
df_movies_cleaned['tmdbId'] = df_movies_cleaned['tmdbId'].astype(int)
df_movies_cleaned['release_year'] = df_movies_cleaned['title'].str.extract(r'\((\d{4})\)')
df_movies_cleaned['clean_title'] = df_movies_cleaned['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True)

print(f"Фильмов с ключом tmdbId: {df_movies_cleaned.shape[0]}")

Файлов для обработки: 23
Книг после фильтрации: 2714


,Name,CountsOfReview,PublishYear,Authors,Rating,Description
2007,"The Hunger Games (The Hunger Games, #1)",154447,2008,Suzanne Collins,4.33,<b>WINNING MEANS FAME AND FORTUNE.<br />LOSING...
1059,"Twilight (Twilight, #1)",94850,2006,Stephenie Meyer,3.59,None
737,The Book Thief,87685,2006,Markus Zusak,4.37,None


Фильмов с ключом tmdbId: 87461


## 3.1. Entity Resolution и Слияние (ETL) — Многослойная верификация

Сопоставление графа знаний Wikidata с датасетом Goodreads с использованием трехуровневой верификации:
1. **Хронологический фильтр**: Исключение из пространства поиска книг, изданных позже релиза экранизации (`PublishYear` $\le$ `release_year`).
2. **Перекрестная валидация (TMDB API)**: Поиск автора книги в списках сценаристов фильма (роли Novel/Author). Совпадение дает повышающий коэффициент достоверности.
3. **Семантическое сходство (NLP)**: Векторизация синопсиса фильма и аннотации книги алгоритмом TF-IDF с последующим вычислением косинусного расстояния (Cosine Similarity) для подтверждения сюжетной идентичности.

In [10]:
%pip install thefuzz

  Using cached thefuzz-0.22.1-py3-none-any.whl.metadata (3.9 kB)
Using cached thefuzz-0.22.1-py3-none-any.whl (8.2 kB)
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 713.4 kB/s eta 0:00:02
   ----------- ---------------------------- 0.5/1.8 MB 713.4 kB/s eta 0:00:02
   ----------------------- ---------------- 1.0/1.8 MB 978.8 kB/s eta 0:00:01
   ----------------------- ---------------- 1.0/1.8 MB 978.8 kB/s eta 0:00:01
   ----------------------- ---------------- 1.0/1.8 MB 978.8 kB/s eta 0:00:01
   ----------------------------------- ---- 1.6/1.8 MB 977.1 kB/s eta 0:00:01
   ----------------------------------- ---- 1.6/1.8 MB 977.1 kB/s et


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\test\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [19]:
TMDB_API_KEY = "90cd5b54a4fc6b7f20b46fad05f1ac88"

In [22]:
import requests
import pandas as pd
import time
import numpy as np
from thefuzz import fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Запуск многослойной системы слияния (Year Filter + TMDB API + NLP)...")

matched_records = []
vec = TfidfVectorizer(stop_words='english')

for idx, row in df_wiki_movies.iterrows():
    tmdb_id = row['tmdbId']
    wiki_source = str(row['source_wiki']).lower()
    wiki_author = str(row['author_wiki']).lower()
    
    movie_year = pd.to_numeric(row['release_year'], errors='coerce')

    movie_overview = ""
    movie_writers = []
    try:
        url = f"https://api.themoviedb.org/3/movie/{tmdb_id}?api_key={TMDB_API_KEY}&append_to_response=credits"
        resp = requests.get(url, timeout=5)
        if resp.status_code == 200:
            data = resp.json()
            movie_overview = data.get('overview', '')
            
            for crew in data.get('credits', {}).get('crew', []):
                if crew.get('department') == 'Writing':
                    movie_writers.append(str(crew.get('name')).lower())
    except Exception:
        pass

    time.sleep(0.05) 

    best_score = 0
    best_match = None

    if pd.notna(movie_year):
        candidate_books = df_books[df_books['PublishYear'] <= movie_year]
    else:
        candidate_books = df_books

    for b_idx, b_row in candidate_books.iterrows():
        gr_book = str(b_row['Name']).lower()
        title_score = fuzz.token_set_ratio(wiki_source, gr_book)

        if title_score > 85:
            gr_author = str(b_row['Authors']).lower()
            book_desc = str(b_row['Description'])

            author_in_crew = any(fuzz.token_set_ratio(gr_author, w) > 85 for w in movie_writers)
            author_wiki_match = fuzz.token_set_ratio(wiki_author, gr_author) > 85 if wiki_author != 'none' else False
            
            author_bonus = 20 if (author_in_crew or author_wiki_match) else 0

            semantic_score = 0
            if movie_overview and book_desc and book_desc != 'None':
                try:
                    tfidf = vec.fit_transform([movie_overview, book_desc])
                    semantic_score = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0] * 100
                except ValueError:
                    pass

            final_score = title_score + author_bonus + (semantic_score * 0.3)

            if final_score > best_score:
                best_score = final_score
                best_match = b_row

    if best_match is not None and best_score >= 85:
        matched_row = row.to_dict()
        for k, v in best_match.to_dict().items():
            matched_row[f"book_{k}"] = v
            
        matched_row['match_score'] = round(best_score, 2)
        matched_records.append(matched_row)

    if (idx + 1) % 10 == 0:
        print(f"Обработано {idx + 1} / {df_wiki_movies.shape[0]} фильмов...")

df_master = pd.DataFrame(matched_records)

print("\nЭтап Entity Resolution завершен")
print(f"Сформировано подтвержденных пар: {df_master.shape[0]}")
if not df_master.empty:
    display(df_master[['title', 'book_Name', 'book_Authors', 'match_score']].sort_values('match_score', ascending=False).head())

Запуск многослойной системы слияния (Year Filter + TMDB API + NLP)...
Обработано 10 / 8960 фильмов...
Обработано 20 / 8960 фильмов...
Обработано 30 / 8960 фильмов...
Обработано 40 / 8960 фильмов...
Обработано 50 / 8960 фильмов...
Обработано 60 / 8960 фильмов...
Обработано 70 / 8960 фильмов...
Обработано 80 / 8960 фильмов...
Обработано 90 / 8960 фильмов...
Обработано 100 / 8960 фильмов...
Обработано 110 / 8960 фильмов...
Обработано 120 / 8960 фильмов...
Обработано 130 / 8960 фильмов...
Обработано 140 / 8960 фильмов...
Обработано 150 / 8960 фильмов...
Обработано 160 / 8960 фильмов...
Обработано 170 / 8960 фильмов...
Обработано 180 / 8960 фильмов...
Обработано 190 / 8960 фильмов...
Обработано 200 / 8960 фильмов...
Обработано 210 / 8960 фильмов...
Обработано 220 / 8960 фильмов...
Обработано 230 / 8960 фильмов...
Обработано 240 / 8960 фильмов...
Обработано 250 / 8960 фильмов...
Обработано 260 / 8960 фильмов...
Обработано 270 / 8960 фильмов...
Обработано 280 / 8960 фильмов...
Обработано 290 

,title,book_Name,book_Authors,match_score
368,Paper Towns (2015),Paper Towns,John Green,145.34
800,Serena (2014),Serena,Ron Rash,130.30
796,Incendiary (2008),Incendiary,Chris Cleave,130.03
330,Vampire Academy (2014),"Shadow Kiss (Vampire Academy, #3)",Richelle Mead,128.75
746,The Guernsey Literary and Potato Peel Pie Soci...,The Guernsey Literary and Potato Peel Pie Society,Mary Ann Shaffer,128.49


## 3.2. Сохранение результатов
Кэширование верифицированного графа знаний (942 франшизы) на локальный диск. Это обеспечивает отказоустойчивость конвейера: при перезапуске ядра дальнейший пайплайн будет загружать готовые связи из CSV-файла, минуя ресурсоемкие этапы обращения к API и вычисления расстояния Левенштейна.

In [23]:
import os
import pandas as pd

checkpoint_file = 'df_master_resolved.csv'

df_master.to_csv(checkpoint_file, index=False, encoding='utf-8')

if os.path.exists(checkpoint_file):
    df_master = pd.read_csv(checkpoint_file)
    file_size = os.path.getsize(checkpoint_file) / 1024
    print(f"Датасет связок успешно загружен с диска: {checkpoint_file} ({file_size:.2f} КБ)")
    print(f"Количество подтвержденных франшиз: {df_master.shape[0]}")
else:
    print("Файл чекпоинта не найден.")

Датасет связок успешно загружен с диска: df_master_resolved.csv (315.35 КБ)
Количество подтвержденных франшиз: 942


## 3.3. Обогащение данных через API (JSON)
Скрипт обращается к The Movie Database (TMDB) API для сбора дат локальных премьер. Для каждого `tmdbId` из основной витрины отправляется GET-запрос к эндпоинту `/movie/{id}/release_dates`. Из полученного JSON извлекаются коды стран (ISO 3166-1) и даты выхода экранизаций. Результат сохраняется в `df_geo_releases.csv` для последующей отрисовки мировой карты на дашборде.

In [28]:
import requests
import pandas as pd
import time
import os

print("Начат сбор географических данных...")

unique_movie_ids = df_master['tmdbId'].unique()
geo_data = []

for count, movie_id in enumerate(unique_movie_ids, 1):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/release_dates?api_key={TMDB_API_KEY}"
    
    try:
        response = requests.get(url, timeout=10)
        
        if response.status_code == 200:
            json_data = response.json()
            
            for country_info in json_data.get('results', []):
                country_code = country_info.get('iso_3166_1')
                release_dates_list = country_info.get('release_dates', [])
                
                if release_dates_list:
                    release_date = release_dates_list[0].get('release_date', '').split('T')[0]
                    geo_data.append({
                        'tmdbId': movie_id,
                        'country_code': country_code,
                        'local_release_date': release_date
                    })
                    
    except Exception:
        pass
        
    time.sleep(0.05) 
    
    if count % 100 == 0:
        print(f"Обработано {count} / {len(unique_movie_ids)} фильмов")

df_geo_releases = pd.DataFrame(geo_data)

checkpoint_filename_geo = 'df_geo_releases.csv'
df_geo_releases.to_csv(checkpoint_filename_geo, index=False, encoding='utf-8')

print("Сбор данных завершен.")
print(f"Сохранено локальных премьер: {df_geo_releases.shape[0]}")
if os.path.exists(checkpoint_filename_geo):
    print(f"Файл: {checkpoint_filename_geo} ({os.path.getsize(checkpoint_filename_geo) / 1024:.2f} КБ)")
    
display(df_geo_releases.head())

Начат сбор географических данных...
Обработано 100 / 947 фильмов
Обработано 200 / 947 фильмов
Обработано 300 / 947 фильмов
Обработано 400 / 947 фильмов
Обработано 500 / 947 фильмов
Обработано 600 / 947 фильмов
Обработано 700 / 947 фильмов
Обработано 800 / 947 фильмов
Обработано 900 / 947 фильмов
Сбор данных завершен.
Сохранено локальных премьер: 20068
Файл: df_geo_releases.csv (416.08 КБ)


,tmdbId,country_code,local_release_date
0,70160,AR,2026-10-14
1,70160,AT,2012-03-22
2,70160,AU,2012-03-22
3,70160,BG,2012-03-23
4,70160,BR,2012-03-23


## 3.4. Обогащение данных: Коммерческие метрики и Сложность лора
Сбор финансовых показателей и метрик вовлеченности фанатского сообщества.
1. **JSON (TMDB API):** Парсинг производственных бюджетов и глобальных кассовых сборов экранизаций.
2. **XML (MediaWiki API):** Автоматическая генерация субдоменов для поиска энциклопедий на платформе Fandom. Извлечение количества статей для оценки объема вымышленной вселенной (Lore Complexity).

In [29]:
import requests
import pandas as pd
import time
import os
import re
import xml.etree.ElementTree as ET

print("Шаг 0. Загрузка витрины данных...")
checkpoint_filename = 'df_master_resolved.csv'

if os.path.exists(checkpoint_filename):
    df_master = pd.read_csv(checkpoint_filename)
    print(f"Франшиз для обработки: {df_master.shape[0]}\n")
else:
    raise FileNotFoundError(f"Файл {checkpoint_filename} не найден.")

print("Шаг 1. Извлечение коммерческих метрик (JSON)...")
financial_data = []
unique_movies = df_master['tmdbId'].unique()

for count, movie_id in enumerate(unique_movies, 1):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={TMDB_API_KEY}"
    try:
        resp = requests.get(url, timeout=5)
        if resp.status_code == 200:
            data = resp.json()
            financial_data.append({
                'tmdbId': movie_id,
                'budget': data.get('budget', 0),
                'revenue': data.get('revenue', 0)
            })
    except Exception:
        pass
        
    time.sleep(0.05)
    if count % 100 == 0:
        print(f"Обработано финансов: {count} / {len(unique_movies)}")

df_financials = pd.DataFrame(financial_data)
if not df_financials.empty:
    df_financials = df_financials[(df_financials['budget'] > 0) & (df_financials['revenue'] > 0)]
    print(f"Собраны финансовые данные для {df_financials.shape[0]} фильмов.\n")

print("Шаг 2. Автоматическая оценка сложности лора (XML)...")
unique_sources = df_master['source_wiki'].dropna().unique()
lore_complexity = []

for count, source in enumerate(unique_sources, 1):
    base_name = re.split(r'[:(]', str(source))[0].strip()
    full_domain = re.sub(r'[^a-z0-9]', '', base_name.lower())
    
    words = base_name.split()
    candidates = [full_domain]
    
    if len(words) > 2:
        short_domain = re.sub(r'[^a-z0-9]', '', ''.join(words[:2])).lower()
        candidates.append(short_domain)
        
    for domain in candidates:
        if not domain: 
            continue
            
        wiki_url = f"https://{domain}.fandom.com/api.php?action=query&meta=siteinfo&siprop=statistics&format=xml"
        try:
            resp = requests.get(wiki_url, timeout=3)
            if resp.status_code == 200:
                root = ET.fromstring(resp.text)
                stats = root.find('.//statistics')
                if stats is not None:
                    lore_complexity.append({
                        'source_wiki': source,
                        'fandom_domain': domain,
                        'articles_count': int(stats.get('articles', 0))
                    })
                    break 
        except Exception:
            pass
            
    if count % 50 == 0:
        print(f"Проверено источников Fandom: {count} / {len(unique_sources)}")

df_lore = pd.DataFrame(lore_complexity)

df_financials.to_csv('df_financials.csv', index=False)
df_lore.to_csv('df_lore.csv', index=False)

print("\nДанные успешно обогащены и сохранены в CSV.")
display(df_financials.head(3))
if not df_lore.empty:
    display(df_lore.sort_values('articles_count', ascending=False).head(3))

Шаг 0. Загрузка витрины данных...
Франшиз для обработки: 947

Шаг 1. Извлечение коммерческих метрик (JSON)...
Обработано финансов: 100 / 947
Обработано финансов: 200 / 947
Обработано финансов: 300 / 947
Обработано финансов: 400 / 947
Обработано финансов: 500 / 947
Обработано финансов: 600 / 947
Обработано финансов: 700 / 947
Обработано финансов: 800 / 947
Обработано финансов: 900 / 947
Собраны финансовые данные для 474 фильмов.

Шаг 2. Автоматическая оценка сложности лора (XML)...
Проверено источников Fandom: 50 / 758
Проверено источников Fandom: 100 / 758
Проверено источников Fandom: 150 / 758
Проверено источников Fandom: 200 / 758
Проверено источников Fandom: 250 / 758
Проверено источников Fandom: 300 / 758
Проверено источников Fandom: 350 / 758
Проверено источников Fandom: 400 / 758
Проверено источников Fandom: 450 / 758
Проверено источников Fandom: 500 / 758
Проверено источников Fandom: 550 / 758
Проверено источников Fandom: 600 / 758
Проверено источников Fandom: 650 / 758
Проверен

,tmdbId,budget,revenue
0,70160,75000000,694000000
1,60304,50000000,226349749
3,542224,5000000,22304357


,source_wiki,fandom_domain,articles_count
106,Character,character,34192
27,Harry Potter,harrypotter,24709
177,Animals,animals,11576


## 3.5. Формирование аналитической витрины (Flat Table)
Слияние разрозненных источников по ключам `tmdbId` и `source_wiki`. 
Синтез целевых метрик:
* **Proxy CAC** = `Бюджет / Сборы * (1 + ln(articles_count))`
* **Organic Acceptance Index** = `avg_sentiment / ln(Сборы)`

In [31]:
import pandas as pd
import numpy as np

print("Шаг 1. Загрузка 4 чекпоинтов...")
df_master = pd.read_csv('df_master_resolved.csv')
df_financials = pd.read_csv('df_financials.csv')
df_lore = pd.read_csv('df_lore.csv')
df_nlp = pd.read_csv('df_nlp_results.csv')

print("Шаг 2. Агрегация NLP-метрик...")
df_nlp_filtered = df_nlp[df_nlp['sentiment_compound'] != 0].copy()
nlp_agg = df_nlp_filtered.groupby('tmdbId').agg(
    avg_sentiment=('sentiment_compound', 'mean'),
    tags_count=('clean_text', 'count')
).reset_index()

print("Шаг 3. Слияние источников (Data Merge)...")
df_final = pd.merge(df_master, df_financials, on='tmdbId', how='inner')

df_lore_clean = df_lore[['source_wiki', 'articles_count']].drop_duplicates(subset=['source_wiki'])
df_final = pd.merge(df_final, df_lore_clean, on='source_wiki', how='left')
df_final['articles_count'] = df_final['articles_count'].fillna(0)

df_final = pd.merge(df_final, nlp_agg, on='tmdbId', how='left')
df_final['avg_sentiment'] = df_final['avg_sentiment'].fillna(0)

print("Шаг 4. Feature Engineering (Синтез бизнес-метрик)...")
df_final['Proxy_CAC'] = (df_final['budget'] / (df_final['revenue'] + 1)) * (1 + np.log1p(df_final['articles_count']))

df_final['Organic_Acceptance_Index'] = df_final['avg_sentiment'] / np.log1p(df_final['revenue'])

columns_to_keep = [
    'tmdbId', 'title', 'book_Name', 'budget', 'revenue', 
    'articles_count', 'avg_sentiment', 'Proxy_CAC', 'Organic_Acceptance_Index'
]
df_final = df_final[[c for c in columns_to_keep if c in df_final.columns]]

df_final.to_csv('df_final_dataset.csv', index=False, encoding='utf-8')
print(f"\nЕдиная аналитическая витрина сформирована (Строк: {df_final.shape[0]}).")
display(df_final.sort_values('Proxy_CAC', ascending=False).head(5))

Шаг 1. Загрузка 4 чекпоинтов...
Шаг 2. Агрегация NLP-метрик...
Шаг 3. Слияние источников (Data Merge)...
Шаг 4. Feature Engineering (Синтез бизнес-метрик)...

Единая аналитическая витрина сформирована (Строк: 474).


,tmdbId,title,book_Name,budget,revenue,articles_count,avg_sentiment,Proxy_CAC,Organic_Acceptance_Index
331,744594,White Noise (2022),White Noise,100000000,71476,30.0,-0.710650,6203.376197,-0.063581
345,338947,Hellraiser (2022),The Hellbound Heart,14000000,10303,0.0,-0.400191,1358.695652,-0.043309
299,408648,Batman & Harley Quinn (2017),Batman: Arkham Asylum - A Serious House on Ser...,3500000,39091,9843.0,0.459080,912.748412,0.043417
234,2728,Postal (2007),"Going Postal (Discworld, #33; Moist von Lipwig...",15000000,146741,983.0,-0.068850,806.683761,-0.005787
222,74254,The House of Usher (2007),The Fall,130000,1677,63.0,0.000000,399.675090,0.000000
